In [6]:
import torch.nn as nn 
import torchvision
import numpy as np
import torch

```
class:
    def __init__(self, in, out, numclass, downsample ...):
        self ....
        def forward
            return
```

In [14]:
class SimpleCNNs(nn.Module):
    def __init__(self, num_class):
        super().__init__()
        self.num_class = num_class # attribute
        # |> methods
        self.feature = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.adap_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(256, num_class)
    def forward(self, x):
        x = self.feature(x)
        x = self.adap_pool(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

model = SimpleCNNs(num_class=5)
dummy_input = torch.randn(3, 3, 224, 224)
output = model(dummy_input)
print(f'output: {output}')

output: tensor([[ 1.1126, -0.1035,  1.2618,  0.5544, -0.2995],
        [ 1.1403, -0.0645,  1.3115,  0.5781, -0.2914],
        [ 1.1520, -0.0691,  1.2179,  0.5487, -0.3063]],
       grad_fn=<AddmmBackward0>)


In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride

        self.features = nn.Sequential(
            nn.Conv2d(self.in_channels, self.out_channels, kernel_size=3, stride=self.stride, padding=1, bias=False),
            nn.BatchNorm2d(self.out_channels),
            nn.ReLU(),

            nn.Conv2d(self.out_channels, self.out_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(self.out_channels)
        )

        self.downsample = None
        if self.stride != 1 or self.in_channels != self.out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, self.out_channels, kernel_size=1, stride=self.stride, bias=False),
                nn.BatchNorm2d(self.out_channels)
            )

    def forward(self, x):
        identity = x
        out = self.features(x)
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity
        out = torch.relu(out)
        return out

In [ ]:
class Resnet18(nn.Module):
    def __init__(self, num_class=1000, in_channels=64):
        super().__init__()
        self.num_class = num_class
        self.in_channels = in_channels

        self.feature = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        
        self.layer_1 = self._make_layers(64, 2, 1)
        self.layer_2 = self._make_layers(128, 2, 2)
        self.layer_3 = self._make_layers(256, 2, 2)
        self.layer_4 = self._make_layers(512, 2, 2)

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, self.num_class)

    def _make_layers(self, output_channels, blocks, stride=1):
        layers = []
        layers.append(BasicBlock(self.in_channels, output_channels, stride))
        self.in_channels = output_channels
        for i in range(1, blocks):
            layers.append(BasicBlock(self.in_channels, output_channels, stride=1))
        return nn.Sequential(*layers)
    def forward(self, x):
        x = self.feature(x)
        x = self.layer_1(x)
        x = self.layer_2(x)
        x = self.layer_3(x)
        x = self.layer_4(x)
        x = self.global_pool(x)
        x = torch.flatten(x, dim = 1)
        x = self.fc(x)

        return x
            